In [1]:
import numpy as np
import torch
import os
import sys
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.insert(0, parent_dir)
from model import AttentionModel
from model_PCA_correlation import AttentionModel_PCA
from model_PCA_cond import ModelPCAcondJ
from dcascore import *
from utils import read_fasta_alignment, remove_duplicate_sequences, add_PCA_coords

# back to original path (in PLM)
sys.path.pop(0)  # Removes the parent_dir from sys.path
from model import AttentionModel

from plm_gen_methods import generate_plm_n_save, generate_coords_n_save, generate_multiple_targets_n_save,generate_plm_vect_n_save
from seq_utils import read_tensor_from_txt, set_seed, letters_to_nums, modify_seq 


In [2]:
"""
    Load Q, K, V matrices from jdoms (after training)
"""
set_seed()
H = 64
d= 10
N = 174
n_epochs = 250
nb_PCA_comp=2
loss_type = 'without_J'
family = 'jdoms' #'jdoms_bacteria_train2'
cwd = parent_dir
Q_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_PCA35_cond/Q_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
K_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_PCA35_cond/K_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
V_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_PCA35_cond/V_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
H,d,N=Q_1.shape
q=V_1.shape[1]


In [14]:
model=ModelPCAcondJ(H,d,N,q,Q=Q_1,V=V_1,K=K_1)
torch.sum(model.Q-Q_1)
device = Q_1.device
L = Q_1.shape[-1]
W=attention_heads_from_model(model,Q_1,K_1,V_1)
print(W.shape)

i_indices = torch.arange(L, device=device).unsqueeze(1)
j_indices = torch.arange(L, device=device).unsqueeze(0)
mask = (i_indices != j_indices).float().unsqueeze(0)  # shape (1, L, L)
# Mask only part without CPA
H, _, M = K_1.shape #(M=N+m)

W[:,:,:-2] = W[:,:,:-2] * mask
    
# Compute Jtens
Jtens = torch.einsum('hri,hab->abri', W, V_1)  # Shape: (q, q, L, L)
q = Jtens.shape[0]
N = Jtens.shape[2]
print(q)
print(N)
print(Jtens.shape)

torch.Size([64, 63, 65])
21
63
torch.Size([21, 56, 63, 65])


In [5]:
print(type(Jtens[0,np.full(100,1),0,0]))

<class 'torch.Tensor'>


In [17]:
save_dir = "generated_vect_sequences_brute_force_condJ"
N_seqs = 30000

n_iter=1500
nb_seq=5000
save_name = f"gen_vect_plm_randinit_n_iter{n_iter}_nb_seq{nb_seq}_Nbins35"
nb_PCA_comp = 2
generate_plm_vect_n_save(save_dir,save_name,Jtens,n_iter,nb_seq,beta=1.2,nb_PCA_comp=nb_PCA_comp,beta_PCA=100,PCA_comp_list=np.array([24+21,20+21]))

100%|██████████| 1500/1500 [06:01<00:00,  4.15it/s]

Generated sequences (letters): ['-YFELFSLPRAFDLDKLARRFRELQRDVHPDRFDSEALALQKSSLINKAYQALKSPFQRALY-', '-YFALFGLPAQLDLQKLKLQYRALQAQFHPDRYQVAAQALQRSSRINEAYQVLKSPLKRAEY-', '-YFELFGLPESLDIDKLKAAFRAAQAQVHPDKFSTAALAQQRTSLINEAYQTLKSPLQRARY-', 'NYFELFGLTESVDEDTLKKRYRKLQAQYHPDKFTRQQEATQKSSQINEAYQVLKNPLQRALH-', 'DYFALFDLNQSLDLELLSARYLQLQRQVHPDKGADLQLALQWASRINEAYQTLKSPLKRAEY-']
Generated sequences saved to generated_vect_sequences_brute_force_condJPCA_comp


In [3]:
H = 64
d= 10
N = 174
n_epochs = 500
nb_PCA_comp=2
loss_type = 'without_J'
family = 'jdoms' #'jdoms_bacteria_train2'
cwd = parent_dir
Q_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_PCA_2models_once_35_bins/Q_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
K_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_PCA_2models_once_35_bins/K_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
V_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_PCA_2models_once_35_bins/V_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
H,d,N=Q_1.shape
q=V_1.shape[1]
model=AttentionModel(H,d,N,q,Q=Q_1,V=V_1,K=K_1)
torch.sum(model.Q-Q_1)
device = Q_1.device
L = Q_1.shape[-1]
W=attention_heads_from_model(model,Q_1,K_1,V_1)
print(W.shape)

i_indices = torch.arange(L, device=device).unsqueeze(1)
j_indices = torch.arange(L, device=device).unsqueeze(0)
mask = (i_indices != j_indices).float().unsqueeze(0)  # shape (1, L, L)
W = W * mask
    
# Compute Jtens
Jtens = torch.einsum('hri,hab->abri', W, V_1)  # Shape: (q, q, L, L)
q = Jtens.shape[0]
N = Jtens.shape[2]
print(q)
print(N)
print(Jtens.shape)
print(Jtens.shape[-1])

torch.Size([64, 63, 63])
21
63
torch.Size([21, 21, 63, 63])
63


In [4]:
Q_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_PCA_2models_once_35_bins/Q_tensor_PCA.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
K_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_PCA_2models_once_35_bins/K_tensor_PCA.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
V_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_PCA_2models_once_35_bins/V_tensor_PCA.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
H,d,N1=Q_1.shape
_,_,N2=K_1.shape
_,q1,q2=V_1.shape
model=AttentionModel_PCA(H,d,N1,N2,q1,q2,Q=Q_1,V=V_1,K=K_1)
torch.sum(model.Q-Q_1)
device = Q_1.device
L = Q_1.shape[-1]
W=attention_heads_from_model(model,Q_1,K_1,V_1)
print(W.shape)

# i_indices = torch.arange(L, device=device).unsqueeze(1)
# j_indices = torch.arange(L, device=device).unsqueeze(0)
# mask = (i_indices != j_indices).float().unsqueeze(0)  # shape (1, L, L)
# W = W * mask
    
# Compute Jtens
Jtens_PCA = torch.einsum('hri,hab->abri', W, V_1)  # Shape: (q, q, L, L)
q = Jtens.shape[0]
N = Jtens.shape[2]
print(q)
print(N)
print(Jtens_PCA.shape)

torch.Size([32, 63, 2])
21
63
torch.Size([21, 35, 63, 2])


In [5]:
save_dir = "generated_vect_sequences_2model_once"
N_seqs = 30000

n_iter=1000
nb_seq=1000
betas=[10]#,0.1,0.5,1,5,10]
for beta in betas:
    save_name = f"gen_vect_plm_randinit_n_iter{n_iter}_nb_seq{nb_seq}_b_PCA_{beta}_b_1_PCA24_20"
    generate_plm_vect_n_save(save_dir,save_name,Jtens,n_iter,nb_seq,beta=1,nb_PCA_comp=nb_PCA_comp,beta_PCA=beta,PCA_comp_list=np.array([24,20]),J_PCA=Jtens_PCA)

100%|██████████| 1000/1000 [01:53<00:00,  8.78it/s]

Generated sequences (letters): ['MSAQFHI-CIYQHKEPHGQKVMIYFILYDVVWSDAVRDLKKFMIKVCLEWNENYKGRIKSW', 'YYIETEQCKQPVDRYINVPSSAQRTSAPENKKLSETSFFQVHMSVNKHRKYMTKCASRPCK', 'DKCAHGFHAPILLKMGKHDNYGAVLNQGYSYAMNLRRLLRELQFGNNFVI-F-SGWNLNSS', 'LFQWSRLKMF-CAQTWSQEVSIWGWIGWMEIY--APCWTKRPGIPQILKHKYPLDWFYIG-', 'VPLLDQESGLFYWEGEVFWAHISLCNLGEPIACWIWETWHGFHWVMKYHALMTRKCAPRWS']
Generated sequences saved to generated_vect_sequences_2model_oncePCA_comp
